# Modélisation Images — Expériences comparatives
## Rakuten France — Classification Multimodale

**Objectif :** Mesurer l'impact du preprocessing image sur les performances de classification.  
Chaque expérience change *uniquement* le preprocessing — le modèle (ResNet50 fine-tuné) reste identique.

| Exp_ID | Preprocessing image | Statut |
|--------|---------------------|--------|
| EXP_I01 | Aucun (baseline) — resize 224×224 + normalisation ImageNet | À faire |
| EXP_I02 | Auto-crop fond blanc → resize 224×224 | À faire |
| EXP_I03 | Filtre flou (Laplacien) — exclure images inutilisables | À faire |
| EXP_I04 | Auto-crop + filtre flou combinés | À faire |
| EXP_I05 | EfficientNet-B4 + meilleure config preprocessing | À faire |
| EXP_I06 | Fine-tuning progressif (3 étapes) | À faire |

**Modèle fixe (EXP_I01–I04) :** ResNet50 pré-entraîné ImageNet  
**Métrique principale :** Weighted F1-score  
**Benchmark officiel :** ResNet50 = 0.5534

---
## 1. Imports et Configuration

In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
import time
import warnings
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights

from PIL import Image
import cv2
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 6)

# ── Détection automatique de l'environnement ──────────────────────────────────
ON_KAGGLE = os.path.exists('/kaggle/input')

if ON_KAGGLE:
    _BASE_INPUT    = '/kaggle/input/datasets/xavierhenry56610/rakuten-france'
    INPUT_PATH     = _BASE_INPUT
    BASE_PATH      = '/kaggle/working'
    PROCESSED_PATH = os.path.join(_BASE_INPUT, 'processed')
    IMG_TRAIN_PATH = os.path.join(_BASE_INPUT, 'image_train', 'image_train')
    IMG_TEST_PATH  = os.path.join(_BASE_INPUT, 'image_test',  'image_test - Copie')
    print('Environnement : Kaggle')
    print(f'IMG_TRAIN_PATH : {IMG_TRAIN_PATH}')
    print(f'IMG_TEST_PATH  : {IMG_TEST_PATH}')
else:
    BASE_PATH      = r'C:\Users\xavie\Desktop\Rakuten'
    PROCESSED_PATH = os.path.join(BASE_PATH, 'processed')
    IMG_TRAIN_PATH = os.path.join(BASE_PATH, 'image_train')
    IMG_TEST_PATH  = os.path.join(BASE_PATH, 'image_test')
    print('Environnement : Local')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('Pas de GPU détecté — entraînement sur CPU sera très lent.')

BATCH_SIZE = 128 if ON_KAGGLE else 64
N_EPOCHS   = 10
LR         = 1e-4
IMG_SIZE   = 224
N_CLASSES  = 27
SEED       = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'BATCH_SIZE : {BATCH_SIZE}')
print('Imports OK')

---
## 2. Chargement des données

In [ ]:
train = pd.read_csv(os.path.join(PROCESSED_PATH, 'train_processed.csv'), index_col=0)
test  = pd.read_csv(os.path.join(PROCESSED_PATH, 'test_processed.csv'),  index_col=0)

with open(os.path.join(PROCESSED_PATH, 'label_encoder.pkl'), 'rb') as f:
    le = pickle.load(f)

def get_img_path(row, folder):
    return os.path.join(folder, f"image_{row['imageid']}_product_{row['productid']}.jpg")

train['img_path'] = train.apply(lambda r: get_img_path(r, IMG_TRAIN_PATH), axis=1)
test['img_path']  = test.apply(lambda r:  get_img_path(r, IMG_TEST_PATH),  axis=1)

# Split 80/20 stratifié — identique aux autres notebooks
X_tr, X_val, y_tr, y_val = train_test_split(
    train, train['label'],
    test_size=0.20, random_state=SEED, stratify=train['label']
)

print(f'Train     : {len(X_tr)}')
print(f'Validation: {len(X_val)}')
print(f'Classes   : {train["label"].nunique()}')

# Poids de classes
classes = np.unique(y_tr)
class_weights = compute_class_weight('balanced', classes=classes, y=y_tr)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print(f'Poids min/max : {class_weights.min():.3f} / {class_weights.max():.3f}')

---
## 3. Analyse qualité des images

Avant de lancer les expériences, on calcule deux métriques de qualité sur un échantillon :
- **white_ratio** : proportion de pixels blancs (fond blanc dominant)
- **blur_score** : variance du Laplacien — faible = image floue

In [ ]:
WHITE_THRESH  = 240    # pixel >= cette valeur sur R,G,B = blanc
BLUR_THRESH   = 100.0  # variance Laplacien < seuil = image floue

def white_ratio(img_path):
    try:
        arr = np.array(Image.open(img_path).convert('RGB'))
        return np.all(arr >= WHITE_THRESH, axis=2).mean()
    except:
        return None

def blur_score(img_path):
    """Variance du Laplacien — plus c'est haut, plus l'image est nette."""
    try:
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            return None
        return cv2.Laplacian(img, cv2.CV_64F).var()
    except:
        return None

# Analyse sur échantillon de 2000 images
SAMPLE_N = 2000
sample_q = train.sample(SAMPLE_N, random_state=SEED).copy()
print(f'Calcul qualité sur {SAMPLE_N} images...')
sample_q['white_ratio'] = sample_q['img_path'].apply(white_ratio)
sample_q['blur_score']  = sample_q['img_path'].apply(blur_score)
sample_q = sample_q.dropna(subset=['white_ratio', 'blur_score'])

n_poorly_cropped = (sample_q['white_ratio'] >= 0.70).sum()
n_blurry         = (sample_q['blur_score']  <  BLUR_THRESH).sum()

print(f'\nImages mal cadrées (>70% blanc) : {n_poorly_cropped} / {len(sample_q)} ({n_poorly_cropped/len(sample_q):.1%})')
print(f'Images floues (Laplacien < {BLUR_THRESH}) : {n_blurry} / {len(sample_q)} ({n_blurry/len(sample_q):.1%})')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(sample_q['white_ratio'], bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(0.70, color='red', linestyle='--', label='Seuil 70%')
axes[0].set_title('Distribution ratio pixels blancs')
axes[0].set_xlabel('Ratio pixels blancs')
axes[0].legend()

axes[1].hist(sample_q['blur_score'].clip(upper=1000), bins=50, color='coral', edgecolor='white')
axes[1].axvline(BLUR_THRESH, color='red', linestyle='--', label=f'Seuil flou = {BLUR_THRESH}')
axes[1].set_title('Distribution score de netteté (variance Laplacien)')
axes[1].set_xlabel('Score netteté (↑ = plus net)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Exemples d'images floues vs nettes
blurry_imgs = sample_q.nsmallest(5, 'blur_score')
sharp_imgs  = sample_q.nlargest(5,  'blur_score')

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
fig.suptitle('Images floues (haut) vs nettes (bas)', fontsize=13)

for j, (_, row) in enumerate(blurry_imgs.iterrows()):
    axes[0][j].imshow(Image.open(row['img_path']))
    axes[0][j].set_title(f'Flou={row["blur_score"]:.0f}\nClasse {row["prdtypecode"]}', fontsize=8)
    axes[0][j].axis('off')

for j, (_, row) in enumerate(sharp_imgs.iterrows()):
    axes[1][j].imshow(Image.open(row['img_path']))
    axes[1][j].set_title(f'Net={row["blur_score"]:.0f}\nClasse {row["prdtypecode"]}', fontsize=8)
    axes[1][j].axis('off')

plt.tight_layout()
plt.show()

---
## 4. Fonctions de preprocessing

In [ ]:
def autocrop_white(img, border_pct=0.02, white_thresh=WHITE_THRESH):
    """
    Recadre automatiquement une PIL Image en supprimant le fond blanc.
    Retourne l'image recadrée (ou l'originale si entièrement blanche).
    """
    arr = np.array(img.convert('RGB'))
    non_white = ~np.all(arr >= white_thresh, axis=2)
    if not non_white.any():
        return img
    rows = np.any(non_white, axis=1)
    cols = np.any(non_white, axis=0)
    r_min, r_max = np.where(rows)[0][[0, -1]]
    c_min, c_max = np.where(cols)[0][[0, -1]]
    h, w = arr.shape[:2]
    border = int(min(h, w) * border_pct)
    r_min = max(0, r_min - border)
    r_max = min(h - 1, r_max + border)
    c_min = max(0, c_min - border)
    c_max = min(w - 1, c_max + border)
    return img.crop((c_min, r_min, c_max + 1, r_max + 1))


def is_usable(img_path, blur_threshold=BLUR_THRESH):
    """
    Retourne True si l'image est suffisamment nette pour être utile.
    Utilisé pour EXP_I03 et EXP_I04.
    """
    score = blur_score(img_path)
    if score is None:
        return False
    return score >= blur_threshold


# Normalisation ImageNet standard
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

base_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

augment_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

print('Fonctions de preprocessing définies.')

---
## 5. Dataset PyTorch et Modèle

In [ ]:
class RakutenImageDataset(Dataset):
    """
    Dataset PyTorch pour les images Rakuten.
    
    Paramètres
    ----------
    df          : DataFrame avec colonnes 'img_path' et optionnellement 'label'
    transform   : transforms à appliquer après chargement
    use_autocrop: appliquer l'auto-recadrage fond blanc avant transform
    blur_filter : si True, remplacer les images floues par une image noire
                  (le modèle apprend à ignorer ces produits → fusion texte)
    """
    def __init__(self, df, transform, use_autocrop=False, blur_filter=False):
        self.df           = df.reset_index(drop=True)
        self.transform    = transform
        self.use_autocrop = use_autocrop
        self.blur_filter  = blur_filter
        self.has_labels   = 'label' in df.columns

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = row['img_path']

        try:
            img = Image.open(img_path).convert('RGB')

            # Filtre flou : remplacer par image noire si trop floue
            if self.blur_filter and not is_usable(img_path):
                img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), color=(0, 0, 0))

            # Auto-crop fond blanc
            elif self.use_autocrop:
                img = autocrop_white(img)

        except Exception:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), color=(128, 128, 128))

        img_tensor = self.transform(img)

        if self.has_labels:
            return img_tensor, int(row['label'])
        return img_tensor


def build_model(n_classes=N_CLASSES, freeze_backbone=False, arch='resnet50'):
    """
    Construit un modèle de classification.

    arch : 'resnet50' ou 'efficientnet_b4'
    freeze_backbone : geler tout sauf la tête (feature extraction)
    """
    if arch == 'resnet50':
        from torchvision.models import resnet50, ResNet50_Weights
        model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
        in_features = model.fc.in_features
        model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_features, n_classes))

    elif arch == 'efficientnet_b4':
        from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
        model = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)
        if freeze_backbone:
            for param in model.features.parameters():
                param.requires_grad = False
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(nn.Dropout(0.4), nn.Linear(in_features, n_classes))

    else:
        raise ValueError(f'Architecture inconnue : {arch}')

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Modèle : {arch} | Params entraînables : {n_params:,}')
    return model.to(DEVICE)


print('Dataset et modèle définis.')

In [ ]:
# Dictionnaire global pour stocker les résultats image
IMG_RESULTS = {}

def train_and_evaluate(exp_id, df_train, df_val, y_train, y_val,
                        use_autocrop=False, blur_filter=False,
                        freeze_backbone=False, augment=True):
    """
    Entraîne ResNet50 et évalue sur la validation.
    Sauvegarde le meilleur modèle (weighted F1) dans models/.
    """
    t0 = time.time()
    print(f'\n{"="*60}')
    print(f'  {exp_id}')
    print(f'  autocrop={use_autocrop} | blur_filter={blur_filter} | freeze={freeze_backbone}')
    print(f'{"="*60}')

    # Datasets
    tr_transform  = augment_transform if augment else base_transform
    ds_train = RakutenImageDataset(df_train, tr_transform,  use_autocrop, blur_filter)
    ds_val   = RakutenImageDataset(df_val,   base_transform, use_autocrop, blur_filter)

    dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=(DEVICE.type == 'cuda'))
    dl_val   = DataLoader(ds_val,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=(DEVICE.type == 'cuda'))

    # Modèle
    model     = build_model(freeze_backbone=freeze_backbone)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                            lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

    os.makedirs(os.path.join(BASE_PATH, 'models'), exist_ok=True)
    best_path = os.path.join(BASE_PATH, 'models', f'{exp_id}_best.pth')

    best_wf1   = 0.0
    history    = {'train_loss': [], 'val_wf1': [], 'val_mf1': []}
    patience   = 3
    no_improve = 0

    for epoch in range(1, N_EPOCHS + 1):
        # ── Train ──
        model.train()
        total_loss = 0.0
        for imgs, labels in dl_train:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(imgs)
        scheduler.step()
        avg_loss = total_loss / len(ds_train)

        # ── Validation ──
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for imgs, labels in dl_val:
                preds = model(imgs.to(DEVICE)).argmax(dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(labels.numpy())

        wf1 = f1_score(all_labels, all_preds, average='weighted')
        mf1 = f1_score(all_labels, all_preds, average='macro')
        history['train_loss'].append(avg_loss)
        history['val_wf1'].append(wf1)
        history['val_mf1'].append(mf1)

        print(f'  Epoch {epoch:2d}/{N_EPOCHS} | Loss={avg_loss:.4f} | '
              f'W-F1={wf1:.4f} | M-F1={mf1:.4f}')

        # Early stopping + sauvegarde du meilleur modèle
        if wf1 > best_wf1:
            best_wf1 = wf1
            torch.save(model.state_dict(), best_path)
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stopping à l\'epoch {epoch}.')
                break

    # Charger le meilleur modèle pour l'évaluation finale
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in dl_val:
            preds = model(imgs.to(DEVICE)).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    final_wf1 = f1_score(all_labels, all_preds, average='weighted')
    final_mf1 = f1_score(all_labels, all_preds, average='macro')
    final_acc = (np.array(all_labels) == np.array(all_preds)).mean()
    duration  = time.time() - t0

    IMG_RESULTS[exp_id] = {
        'weighted_f1': final_wf1,
        'macro_f1':    final_mf1,
        'accuracy':    final_acc,
        'duration_s':  duration,
        'history':     history,
        'y_pred':      np.array(all_preds),
        'y_true':      np.array(all_labels),
        'model_path':  best_path
    }

    print(f'\n  ✓ Meilleur W-F1 : {final_wf1:.4f} | M-F1 : {final_mf1:.4f}')
    print(f'  Durée totale   : {duration/60:.1f} min')
    return IMG_RESULTS[exp_id]


print('Fonction d\'entraînement définie.')

---
## EXP_I01 — Baseline (aucun preprocessing)

ResNet50 ImageNet → simple resize 224×224 + normalisation.  
**Référence :** benchmark officiel = 0.5534

In [ ]:
res_i01 = train_and_evaluate(
    exp_id       = 'EXP_I01',
    df_train     = X_tr,
    df_val       = X_val,
    y_train      = y_tr,
    y_val        = y_val,
    use_autocrop = False,
    blur_filter  = False
)

---
## EXP_I02 — Auto-crop fond blanc

Recadrage automatique sur les pixels non-blancs avant resize.  
Le produit occupe une fraction plus grande de l'image → meilleur signal visuel.  
48% des images bénéficient de cette transformation (fond blanc > 70%).

In [ ]:
res_i02 = train_and_evaluate(
    exp_id       = 'EXP_I02',
    df_train     = X_tr,
    df_val       = X_val,
    y_train      = y_tr,
    y_val        = y_val,
    use_autocrop = True,
    blur_filter  = False
)

---
## EXP_I03 — Filtre flou (images inutilisables → image noire)

Les images trop floues (variance Laplacien < 100) sont remplacées par une image noire.  
Le modèle apprend à ignorer ces inputs → lors de la fusion, le texte prend le relai pour ces produits.

> **Note :** on ne supprime pas ces produits — on doit quand même les classifier.  
> L'image noire force le modèle à produire une prédiction incertaine,  
> ce qui favorisera la confiance au texte lors de la fusion.

In [ ]:
res_i03 = train_and_evaluate(
    exp_id       = 'EXP_I03',
    df_train     = X_tr,
    df_val       = X_val,
    y_train      = y_tr,
    y_val        = y_val,
    use_autocrop = False,
    blur_filter  = True
)

---
## EXP_I04 — Auto-crop + filtre flou (combiné)

Meilleure configuration attendue : auto-recadrage ET exclusion des images floues.

In [ ]:
res_i04 = train_and_evaluate(
    exp_id       = 'EXP_I04',
    df_train     = X_tr,
    df_val       = X_val,
    y_train      = y_tr,
    y_val        = y_val,
    use_autocrop = True,
    blur_filter  = True
)

---
## EXP_I05 — EfficientNet-B4

**Pourquoi EfficientNet-B4 ?**

| Modèle | Params | ImageNet Top-1 | Gain vs ResNet50 |
|--------|--------|----------------|------------------|
| ResNet50 | 25M | 76.1% | — |
| EfficientNet-B4 | 19M | 83.4% | +7.3 pp |

Moins de paramètres, meilleure précision. EfficientNet optimise simultanément profondeur, largeur et résolution via un coefficient de mise à l'échelle composé (compound scaling).

On utilise le **meilleur preprocessing trouvé dans EXP_I01–I04** (à adapter selon vos résultats).

In [ ]:
# Résultats EXP_I01-I04 → baseline sans preprocessing est le meilleur
BEST_AUTOCROP = False
BEST_BLUR     = False

# Vider le cache GPU avant EfficientNet (plus gourmand que ResNet50)
torch.cuda.empty_cache()

# Batch size réduit pour EfficientNet-B4 (évite OOM sur T4 16GB)
EFF_BATCH_SIZE = 48

ds_tr_eff  = RakutenImageDataset(X_tr,  augment_transform, BEST_AUTOCROP, BEST_BLUR)
ds_val_eff = RakutenImageDataset(X_val, base_transform,    BEST_AUTOCROP, BEST_BLUR)

dl_tr_eff  = DataLoader(ds_tr_eff,  batch_size=EFF_BATCH_SIZE, shuffle=True,
                        num_workers=2, pin_memory=True)
dl_val_eff = DataLoader(ds_val_eff, batch_size=EFF_BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)

model_eff = build_model(arch='efficientnet_b4')
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.AdamW(model_eff.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

best_wf1_eff = 0.0
best_path_eff = os.path.join(BASE_PATH, 'models', 'EXP_I05_best.pth')
os.makedirs(os.path.join(BASE_PATH, 'models'), exist_ok=True)
history_eff   = {'train_loss': [], 'val_wf1': [], 'val_mf1': []}
patience, no_improve = 3, 0

print(f'\n{"="*60}')
print(f'  EXP_I05 — EfficientNet-B4')
print(f'  autocrop={BEST_AUTOCROP} | blur_filter={BEST_BLUR} | batch={EFF_BATCH_SIZE}')
print(f'{"="*60}')

t0 = time.time()
for epoch in range(1, N_EPOCHS + 1):
    model_eff.train()
    total_loss = 0.0
    for imgs, labels in dl_tr_eff:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model_eff(imgs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(imgs)
    scheduler.step()
    avg_loss = total_loss / len(ds_tr_eff)

    model_eff.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in dl_val_eff:
            preds = model_eff(imgs.to(DEVICE)).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    wf1 = f1_score(all_labels, all_preds, average='weighted')
    mf1 = f1_score(all_labels, all_preds, average='macro')
    history_eff['train_loss'].append(avg_loss)
    history_eff['val_wf1'].append(wf1)
    history_eff['val_mf1'].append(mf1)
    print(f'  Epoch {epoch:2d}/{N_EPOCHS} | Loss={avg_loss:.4f} | W-F1={wf1:.4f} | M-F1={mf1:.4f}')

    if wf1 > best_wf1_eff:
        best_wf1_eff = wf1
        torch.save(model_eff.state_dict(), best_path_eff)
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f'  Early stopping à l\'epoch {epoch}.')
            break

# Évaluation finale
model_eff.load_state_dict(torch.load(best_path_eff, map_location=DEVICE))
model_eff.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in dl_val_eff:
        preds = model_eff(imgs.to(DEVICE)).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

final_wf1 = f1_score(all_labels, all_preds, average='weighted')
final_mf1 = f1_score(all_labels, all_preds, average='macro')

IMG_RESULTS['EXP_I05'] = {
    'weighted_f1': final_wf1, 'macro_f1': final_mf1,
    'accuracy': (np.array(all_labels) == np.array(all_preds)).mean(),
    'duration_s': time.time() - t0, 'history': history_eff,
    'y_pred': np.array(all_preds), 'y_true': np.array(all_labels),
    'model_path': best_path_eff
}
print(f'\n  ✓ EfficientNet-B4 — W-F1={final_wf1:.4f} | M-F1={final_mf1:.4f}')
print(f'  Meilleure ResNet50 (EXP_I01) : 0.6774')
print(f'  Delta EfficientNet vs ResNet50 : {final_wf1 - 0.6774:+.4f}')

---
## EXP_I06 — Fine-tuning progressif (3 étapes)

**Problème du fine-tuning classique :** dégeler tous les poids dès le début écrase les représentations ImageNet apprises, surtout avec un LR trop élevé.

**Solution — 3 étapes :**

```
Étape 1 (5 epochs, LR=1e-3)  : backbone gelé → entraîner seulement la tête FC
                                 Le modèle apprend les classes Rakuten sans abîmer ImageNet
Étape 2 (5 epochs, LR=1e-4)  : dégeler le dernier bloc ResNet (layer4)
                                 Adapter les features de haut niveau au domaine
Étape 3 (5 epochs, LR=1e-5)  : dégeler tout le backbone
                                 Fine-tune global avec LR très faible
```

Gain typique : **+0.03 à +0.05** vs fine-tuning classique.  
On part du meilleur modèle (EfficientNet-B4 si EXP_I05 > EXP_I04, sinon ResNet50).

In [ ]:
def run_stage(model, dl_train, dl_val, criterion, n_epochs, lr, stage_name,
              best_wf1, best_path, history):
    """Entraîne un modèle pour une étape du fine-tuning progressif."""
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                            lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    print(f'\n  ── {stage_name} (LR={lr}, {n_epochs} epochs) ──')
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'     Params entraînables : {n_trainable:,}')

    patience, no_improve = 2, 0
    for epoch in range(1, n_epochs + 1):
        model.train()
        total_loss = 0.0
        for imgs, labels in dl_train:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(imgs)
        scheduler.step()

        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for imgs, labels in dl_val:
                preds = model(imgs.to(DEVICE)).argmax(dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(labels.numpy())

        wf1 = f1_score(all_labels, all_preds, average='weighted')
        mf1 = f1_score(all_labels, all_preds, average='macro')
        history['val_wf1'].append(wf1)
        history['val_mf1'].append(mf1)
        history['train_loss'].append(total_loss / len(dl_train.dataset))
        print(f'     Epoch {epoch}/{n_epochs} | Loss={total_loss/len(dl_train.dataset):.4f} '
              f'| W-F1={wf1:.4f} | M-F1={mf1:.4f}')

        if wf1 > best_wf1:
            best_wf1 = wf1
            torch.save(model.state_dict(), best_path)
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'     Early stopping.')
                break
    return best_wf1


# ── Setup ─────────────────────────────────────────────────────────────────────
# Utiliser EfficientNet-B4 si disponible et meilleur, sinon ResNet50
use_effnet = 'EXP_I05' in IMG_RESULTS and \
             IMG_RESULTS['EXP_I05']['weighted_f1'] > IMG_RESULTS.get('EXP_I04', {}).get('weighted_f1', 0)
arch = 'efficientnet_b4' if use_effnet else 'resnet50'
print(f'Architecture choisie pour EXP_I06 : {arch}')

ds_tr_prog  = RakutenImageDataset(X_tr,  augment_transform, BEST_AUTOCROP, BEST_BLUR)
ds_val_prog = RakutenImageDataset(X_val, base_transform,    BEST_AUTOCROP, BEST_BLUR)
dl_tr_prog  = DataLoader(ds_tr_prog,  batch_size=BATCH_SIZE, shuffle=True,
                         num_workers=2, pin_memory=(DEVICE.type == 'cuda'))
dl_val_prog = DataLoader(ds_val_prog, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=(DEVICE.type == 'cuda'))

# Partir d'un backbone complètement gelé
model_prog = build_model(arch=arch, freeze_backbone=True)
criterion  = nn.CrossEntropyLoss(weight=class_weights_tensor)
best_path_prog = os.path.join(BASE_PATH, 'models', 'EXP_I06_best.pth')
history_prog   = {'train_loss': [], 'val_wf1': [], 'val_mf1': []}
best_wf1_prog  = 0.0

t0 = time.time()
print(f'\n{"="*60}')
print(f'  EXP_I06 — Fine-tuning progressif ({arch})')
print(f'{"="*60}')

# ── Étape 1 : tête seulement ──────────────────────────────────────────────────
best_wf1_prog = run_stage(
    model_prog, dl_tr_prog, dl_val_prog, criterion,
    n_epochs=5, lr=1e-3, stage_name='Étape 1 — Tête FC seule',
    best_wf1=best_wf1_prog, best_path=best_path_prog, history=history_prog
)

# ── Étape 2 : dégeler le dernier bloc ────────────────────────────────────────
if arch == 'resnet50':
    for param in model_prog.layer4.parameters():
        param.requires_grad = True
elif arch == 'efficientnet_b4':
    for param in model_prog.features[-2:].parameters():
        param.requires_grad = True

best_wf1_prog = run_stage(
    model_prog, dl_tr_prog, dl_val_prog, criterion,
    n_epochs=5, lr=1e-4, stage_name='Étape 2 — Dernier bloc dégelé',
    best_wf1=best_wf1_prog, best_path=best_path_prog, history=history_prog
)

# ── Étape 3 : dégeler tout ───────────────────────────────────────────────────
for param in model_prog.parameters():
    param.requires_grad = True

best_wf1_prog = run_stage(
    model_prog, dl_tr_prog, dl_val_prog, criterion,
    n_epochs=5, lr=1e-5, stage_name='Étape 3 — Backbone complet dégelé',
    best_wf1=best_wf1_prog, best_path=best_path_prog, history=history_prog
)

# ── Évaluation finale ─────────────────────────────────────────────────────────
model_prog.load_state_dict(torch.load(best_path_prog, map_location=DEVICE))
model_prog.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in dl_val_prog:
        preds = model_prog(imgs.to(DEVICE)).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

final_wf1 = f1_score(all_labels, all_preds, average='weighted')
final_mf1 = f1_score(all_labels, all_preds, average='macro')

IMG_RESULTS['EXP_I06'] = {
    'weighted_f1': final_wf1, 'macro_f1': final_mf1,
    'accuracy': (np.array(all_labels) == np.array(all_preds)).mean(),
    'duration_s': time.time() - t0, 'history': history_prog,
    'y_pred': np.array(all_preds), 'y_true': np.array(all_labels),
    'model_path': best_path_prog
}

print(f'\n  ✓ Fine-tuning progressif — W-F1={final_wf1:.4f} | M-F1={final_mf1:.4f}')
print(f'  Durée totale : {(time.time()-t0)/60:.1f} min')

# Courbe d'apprentissage par étape
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(history_prog['val_wf1'], marker='o', markersize=4)
ax.axvline(4,  color='orange', linestyle='--', alpha=0.7, label='Fin étape 1')
ax.axvline(9,  color='red',    linestyle='--', alpha=0.7, label='Fin étape 2')
ax.set_title('Fine-tuning progressif — Weighted F1 par epoch')
ax.set_xlabel('Epoch')
ax.set_ylabel('Weighted F1')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Grad-CAM — Interprétabilité visuelle

**Qu'est-ce que Grad-CAM ?**  
Grad-CAM (*Gradient-weighted Class Activation Mapping*) calcule, pour une prédiction donnée, quelles zones de l'image ont le plus contribué à la décision du modèle. Il utilise les gradients de la classe prédite par rapport aux feature maps de la dernière couche convolutive.

**Utilité pour ce projet :**
- Vérifier que le modèle regarde bien le produit (et pas le fond blanc)
- Comparer l'attention avant/après autocrop
- Identifier pourquoi certaines classes sont difficiles (livres, BDs → couvertures similaires)
- Figures directement exploitables dans le rapport de modélisation

In [ ]:
import torch.nn.functional as F

class GradCAM:
    """
    Implémentation manuelle de Grad-CAM pour ResNet50 et EfficientNet.
    Pas de dépendance externe.
    """
    def __init__(self, model, arch='resnet50'):
        self.model    = model
        self.arch     = arch
        self.gradients = None
        self.activations = None
        self._register_hooks()

    def _register_hooks(self):
        # Couche cible : dernière couche convolutive
        if self.arch == 'resnet50':
            target_layer = self.model.layer4[-1].conv3
        elif self.arch == 'efficientnet_b4':
            target_layer = self.model.features[-1][0]
        else:
            raise ValueError(f'Arch inconnue : {self.arch}')

        target_layer.register_forward_hook(
            lambda m, inp, out: setattr(self, 'activations', out.detach())
        )
        target_layer.register_full_backward_hook(
            lambda m, grad_in, grad_out: setattr(self, 'gradients', grad_out[0].detach())
        )

    def __call__(self, img_tensor, class_idx=None):
        """
        img_tensor : (1, 3, H, W) sur DEVICE
        class_idx  : classe cible (None = classe prédite)
        Retourne : (heatmap_np 0–1, class_idx, confidence)
        """
        self.model.eval()
        logits = self.model(img_tensor)
        probs  = torch.softmax(logits, dim=1)

        if class_idx is None:
            class_idx = logits.argmax(dim=1).item()

        self.model.zero_grad()
        logits[0, class_idx].backward()

        # Pondération des feature maps par les gradients moyennés
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)  # (1, C, 1, 1)
        cam     = (weights * self.activations).sum(dim=1, keepdim=True)  # (1, 1, h, w)
        cam     = F.relu(cam)
        cam     = F.interpolate(cam, size=(IMG_SIZE, IMG_SIZE),
                                mode='bilinear', align_corners=False)
        cam     = cam.squeeze().cpu().numpy()
        cam    -= cam.min()
        if cam.max() > 0:
            cam /= cam.max()

        confidence = probs[0, class_idx].item()
        return cam, class_idx, confidence


def show_gradcam(model, arch, img_paths, labels_true, class_names,
                 use_autocrop=False, n_cols=4, title='Grad-CAM'):
    """
    Affiche n images avec leur heatmap Grad-CAM superposée.
    """
    gcam = GradCAM(model, arch=arch)
    n    = len(img_paths)
    fig, axes = plt.subplots(n, 2, figsize=(10, n * 4))
    if n == 1:
        axes = [axes]
    fig.suptitle(title, fontsize=13)

    for i, (img_path, true_label) in enumerate(zip(img_paths, labels_true)):
        img_pil = Image.open(img_path).convert('RGB')
        if use_autocrop:
            img_pil = autocrop_white(img_pil)

        img_tensor = base_transform(img_pil).unsqueeze(0).to(DEVICE)
        cam, pred_idx, conf = gcam(img_tensor)

        # Superposition heatmap
        import cv2 as _cv2
        img_np  = np.array(img_pil.resize((IMG_SIZE, IMG_SIZE)))
        heatmap = _cv2.applyColorMap(np.uint8(255 * cam), _cv2.COLORMAP_JET)
        heatmap = _cv2.cvtColor(heatmap, _cv2.COLOR_BGR2RGB)
        overlay = (0.5 * img_np + 0.5 * heatmap).astype(np.uint8)

        axes[i][0].imshow(img_np)
        axes[i][0].set_title(f'Réelle : {class_names.get(true_label, true_label)}', fontsize=9)
        axes[i][0].axis('off')

        color = 'green' if pred_idx == true_label else 'red'
        axes[i][1].imshow(overlay)
        axes[i][1].set_title(
            f'Prédite : {class_names.get(pred_idx, pred_idx)} ({conf:.1%})',
            fontsize=9, color=color
        )
        axes[i][1].axis('off')

    plt.tight_layout()
    return fig


CLASS_NAMES_INV = {
    0:'Livres', 1:'Jeux vidéo PC', 2:'Access. gaming', 3:'Consoles',
    4:'Dérivés geek', 5:'Cartes collec.', 6:'Figurines JS', 7:'Jouets',
    8:'Jeux société', 9:'Petites voitures', 10:'Access. enfance',
    11:'Jeux extérieur', 12:'Sacs/access.', 13:'Mobilier', 14:'Linge maison',
    15:'Nourriture', 16:'Lampes/déco', 17:'Access. animaux', 18:'Magazines',
    19:'Livres & BDs', 20:'Consoles/jeux', 21:'Papeterie', 22:'Mobilier ext.',
    23:'Piscines/Spas', 24:'Outillage', 25:'BDs & livres', 26:'Jeux DL'
}

print('Grad-CAM défini.')

In [ ]:
# ── Grad-CAM 1 : 5 exemples bien classifiés ──────────────────────────────────
# Utiliser le meilleur modèle disponible
best_exp_id = max(IMG_RESULTS, key=lambda k: IMG_RESULTS[k]['weighted_f1'])
best_model_path = IMG_RESULTS[best_exp_id]['model_path']
best_arch = 'efficientnet_b4' if 'I05' in best_exp_id or 'I06' in best_exp_id else 'resnet50'

print(f'Meilleur modèle pour Grad-CAM : {best_exp_id} ({best_arch})')
model_gcam = build_model(arch=best_arch)
model_gcam.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model_gcam.eval()

# Sélectionner 5 produits bien classifiés (pred == true)
correct_mask = IMG_RESULTS[best_exp_id]['y_pred'] == IMG_RESULTS[best_exp_id]['y_true']
correct_idx  = np.where(correct_mask)[0][:5]
sample_val   = X_val.reset_index(drop=True).iloc[correct_idx]

fig = show_gradcam(
    model_gcam, best_arch,
    img_paths   = sample_val['img_path'].tolist(),
    labels_true = IMG_RESULTS[best_exp_id]['y_true'][correct_idx].tolist(),
    class_names = CLASS_NAMES_INV,
    use_autocrop= BEST_AUTOCROP,
    title       = f'Grad-CAM — Prédictions correctes ({best_exp_id})'
)
os.makedirs(os.path.join(BASE_PATH, 'docs', 'figures'), exist_ok=True)
fig.savefig(os.path.join(BASE_PATH, 'docs', 'figures', 'gradcam_correct.png'),
            dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Grad-CAM 2 : classes difficiles (Livres vs BDs — confusion fréquente) ─────
# Classes 0 (Livres), 19 (Livres & BDs), 25 (BDs & livres) — souvent confondues
HARD_CLASSES = [0, 19, 25]

hard_samples = []
for cls in HARD_CLASSES:
    mask = (IMG_RESULTS[best_exp_id]['y_true'] == cls)
    idxs = np.where(mask)[0][:2]
    for idx in idxs:
        hard_samples.append((
            X_val.reset_index(drop=True).iloc[idx]['img_path'],
            int(IMG_RESULTS[best_exp_id]['y_true'][idx])
        ))

img_paths_hard  = [s[0] for s in hard_samples]
labels_hard     = [s[1] for s in hard_samples]

fig2 = show_gradcam(
    model_gcam, best_arch,
    img_paths   = img_paths_hard,
    labels_true = labels_hard,
    class_names = CLASS_NAMES_INV,
    use_autocrop= BEST_AUTOCROP,
    title       = 'Grad-CAM — Classes difficiles : Livres / BDs (confusion fréquente)'
)
fig2.savefig(os.path.join(BASE_PATH, 'docs', 'figures', 'gradcam_hard_classes.png'),
             dpi=120, bbox_inches='tight')
plt.show()

print('\nConclusion Grad-CAM :')
print('  → Zones rouges/jaunes = zones les plus discriminantes pour le modèle')
print('  → Si le modèle regarde le fond plutôt que le produit → autocrop nécessaire')
print('  → Figures sauvegardées dans docs/figures/')

---
## Comparaison des résultats

In [ ]:
# Ajouter le benchmark officiel
IMG_RESULTS['BENCHMARK'] = {
    'weighted_f1': 0.5534, 'macro_f1': None, 'accuracy': None,
    'duration_s': None, 'history': None
}

labels_map = {
    'EXP_I01':   'Baseline (resize seulement)',
    'EXP_I02':   'Auto-crop fond blanc',
    'EXP_I03':   'Filtre flou',
    'EXP_I04':   'Auto-crop + filtre flou',
    'BENCHMARK': 'Benchmark officiel (ResNet50)',
}

rows = []
for exp_id, res in IMG_RESULTS.items():
    rows.append({
        'Exp_ID':      exp_id,
        'Description': labels_map.get(exp_id, exp_id),
        'Weighted F1': round(res['weighted_f1'], 4),
        'Macro F1':    round(res['macro_f1'], 4) if res['macro_f1'] else None,
        'Durée (min)': round(res['duration_s'] / 60, 1) if res['duration_s'] else None,
    })

df_img = pd.DataFrame(rows).sort_values('Weighted F1', ascending=False).reset_index(drop=True)

display(df_img.style
    .background_gradient(subset=['Weighted F1'], cmap='RdYlGn')
    .format({'Weighted F1': '{:.4f}', 'Macro F1': '{:.4f}'})
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Comparaison expériences image — ResNet50', fontsize=14)

colors = ['#9E9E9E' if e == 'BENCHMARK' else '#2196F3' for e in df_img['Exp_ID']]
axes[0].barh(df_img['Description'], df_img['Weighted F1'], color=colors)
axes[0].axvline(0.5534, color='orange', linestyle='--', label='Benchmark (0.5534)')
axes[0].set_title('Weighted F1-score')
axes[0].set_xlabel('F1 score')
axes[0].legend()
axes[0].set_xlim(0.4, 0.9)

# Courbes d'apprentissage
for exp_id, res in IMG_RESULTS.items():
    if res.get('history'):
        axes[1].plot(res['history']['val_wf1'], label=exp_id, marker='o', markersize=4)
axes[1].set_title('Weighted F1 en validation par epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Weighted F1')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs(os.path.join(BASE_PATH, 'docs', 'figures'), exist_ok=True)
plt.savefig(os.path.join(BASE_PATH, 'docs', 'figures', 'mod_image_experiments.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# F1 par classe — meilleure expérience image
best_exp = df_img[df_img['Exp_ID'] != 'BENCHMARK'].iloc[0]['Exp_ID']
res_best = IMG_RESULTS[best_exp]

CLASS_NAMES = {
    0:'Livres', 1:'Jeux vidéo PC', 2:'Access. gaming', 3:'Consoles',
    4:'Dérivés geek', 5:'Cartes collec.', 6:'Figurines JS', 7:'Jouets',
    8:'Jeux société', 9:'Petites voitures', 10:'Access. enfance',
    11:'Jeux extérieur', 12:'Sacs/access.', 13:'Mobilier', 14:'Linge maison',
    15:'Nourriture', 16:'Lampes/déco', 17:'Access. animaux', 18:'Magazines',
    19:'Livres & BDs', 20:'Consoles/jeux', 21:'Papeterie', 22:'Mobilier ext.',
    23:'Piscines/Spas', 24:'Outillage', 25:'BDs & livres', 26:'Jeux DL'
}

f1_per_class = f1_score(res_best['y_true'], res_best['y_pred'], average=None)
df_f1 = pd.DataFrame({
    'Classe': [CLASS_NAMES[i] for i in range(N_CLASSES)],
    'F1 Image': f1_per_class
}).sort_values('F1 Image')

fig, ax = plt.subplots(figsize=(10, 9))
colors_bar = ['#F44336' if f < 0.5 else '#FF9800' if f < 0.7 else '#4CAF50'
              for f in df_f1['F1 Image']]
ax.barh(df_f1['Classe'], df_f1['F1 Image'], color=colors_bar)
ax.axvline(0.5, color='red',    linestyle='--', alpha=0.5)
ax.axvline(0.7, color='orange', linestyle='--', alpha=0.5)
ax.set_title(f'F1 par classe — {best_exp}')
ax.set_xlabel('F1 score')
plt.tight_layout()
plt.show()

print(f'\nClasses difficiles (F1 < 0.50) :')
display(df_f1[df_f1['F1 Image'] < 0.50])

In [ ]:
# Sauvegarde des résultats
df_img.to_csv(os.path.join(PROCESSED_PATH, 'image_experiments_results.csv'), index=False)
print('Résultats sauvegardés : processed/image_experiments_results.csv')
print(f'\nMeilleure config image : {best_exp} — W-F1 = {IMG_RESULTS[best_exp]["weighted_f1"]:.4f}')
print(f'Vs benchmark officiel  : {IMG_RESULTS[best_exp]["weighted_f1"] - 0.5534:+.4f}')

---
## Conclusions

*(À compléter après exécution)*

| Question | Résultat observé |
|----------|------------------|
| Auto-crop améliore-t-il ? | ? |
| Le filtre flou aide-t-il ? | ? |
| Meilleure config image | ? |
| Gap image vs texte (0.82) | Attendu : image ~0.60-0.65 après preprocessing |

**Prochain notebook :** `modelisation_fusion.ipynb` — combinaison texte + image (late fusion).

**Stratégie de fusion envisagée :**  
Pour chaque produit, on dispose de deux vecteurs de probabilités (27 classes) :  
- `proba_text[i]` — issu du meilleur modèle texte  
- `proba_image[i]` — issu du meilleur modèle image  

Fusion pondérée adaptative :
```
proba_final = α(i) * proba_text + (1 - α(i)) * proba_image
```
où `α(i)` est plus grand quand `has_desc=0` ou quand l'image est floue.